In [ ]:
from cryoswath.misc import rgi_code_translator, _norm_isf_25, dem_path, l3_path
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from rasterio.warp import Resampling
from rioxarray.exceptions import NoDataInBounds, OneDimensionalRaster
import seaborn as sns
from shapely.geometry import box
from cryoswath_paper_workflow.workflow_support import snakemake_fallback_builder, o1o2reg_tuple_list, strip_spec_chars, project_path
import xarray as xr

snakemake_fallback = snakemake_fallback_builder(globals())

In [ ]:
df = pd.concat(
    [
        pd.read_csv(
            snakemake_fallback(
                "input",
                input_idx,
                str(project_path("resources", f"calval__compare_arcticdem_strips__interpolation_type__{rgi_code_translator(f'{o1:02d}', out_type='long_code')}.csv")),
            ),
            index_col=["strip", "intersec_idx", "flag"],
        )
        for input_idx, (o1, _) in enumerate(o1o2reg_tuple_list)
    ],
    keys=[o1 for o1, _ in o1o2reg_tuple_list],
)
df.rename_axis(["o1", *df.index.names[1:]], inplace=True)
# The theoretical case of a strip interecting multiple regions didn't occur. Drop its index.
if len(df.index.levels[2]) == 1:
    df = df.droplevel(2)

In [ ]:
print(df.describe())
outlier = df._median < -25
df[outlier]

In [ ]:
missing_strip_paths = []
for o1, item_id, _ in df[outlier].index:
    strip_path = Path(dem_path, "strips") / (item_id + "__32m_mean.tif")
    if not strip_path.exists():
        missing_strip_paths.append(strip_path)
        continue

    strip = xr.open_dataarray(strip_path, chunks={})
    for file in Path(l3_path).glob(
        "__".join(["*", strip_spec_chars(rgi_code_translator(f"{o1:02d}")), "*.nc"])
    ):
        ds = xr.open_dataset(file, chunks={}).isel(time=0)
        if box(*strip.rio.bounds()).intersects(
            box(*ds.rio.transform_bounds(strip.rio.crs))
        ):
            try:
                clip = ds.rio.clip_box(
                    *box(*strip.rio.transform_bounds(ds.rio.crs)).bounds
                )
            except (NoDataInBounds, OneDimensionalRaster):
                continue
            strip_repr = (
                strip.rio.reproject_match(clip, resampling=Resampling.average)
                - clip.elev_diff_ref
            )
            strip_repr.plot(robust=True)
            plt.title(f'"Strip" vs "Mosaic"  ({strip_repr.rio.crs})')
            plt.show()

if missing_strip_paths:
    print(
        "Skipped "
        f"{len(missing_strip_paths)} outlier strip diagnostic plot(s) because "
        "coarsened ArcticDEM strip files are missing."
    )
    for path in missing_strip_paths[:10]:
        print(f"  {path}")
    if len(missing_strip_paths) > 10:
        print(f"  ... and {len(missing_strip_paths) - 10} more")


There are single outliers with median deviation from ArcticDEM of less
than -25 m. When the coarsened ArcticDEM strip files are available, the
diagnostic plots above compare those outliers against the monthly mosaic.
Only a patch of elevation estimates are offset with regards to the ArcticDEM
Mosaic product. This could be due to an undetected issue. The outliers are
excluded from the analysis.


In [ ]:
df = df[~outlier]

Below, it is inspected whether there are strong error correltations that
would need to be accounted for.

In [ ]:
df["_date"] = pd.to_datetime(
    df.index.get_level_values("strip").str.extract(r".*_(20\d{6})_.*").iloc(1)[0]
).values
df["month"] = df._date.dt.month
df["sqrt_count"] = df._count**0.5

In [ ]:
rng = np.random.default_rng(0)

In [ ]:
for x, y in combinations(["o1", "month", "flag", "_date"], 2):
    for stat in ["_median", "_iqr"]:
        sns.relplot(
            data=df.reset_index().apply(
                lambda col: (
                    col
                    if not pd.api.types.is_integer_dtype(col)
                    else col + rng.uniform(-0.3, 0.3, len(col))
                )
            ),
            x=x,
            y=y,
            hue=stat,
            size="sqrt_count",
            palette=("vlag" if stat[1] == "m" else "viridis"),
        )
        plt.show()

In [ ]:
diff = df._median.groupby(level=["o1", "strip"], group_keys=False).apply(
    lambda x: x - x.iloc[0]
)
_diff = diff.rename("bias").drop(0, level="flag").reset_index()
flag_order = sorted(_diff["flag"].unique())
_diff["flag"] = pd.Categorical(_diff["flag"], categories=flag_order, ordered=True)

plt.figure(figsize=(12 / 2.54, 12 / 2.54 * 16 / 16))

sns.boxenplot(
    data=_diff[_diff["bias"] > -90],
    x="bias",
    y="flag",
    hue="o1",
    palette="tab10",
    showfliers=False,
)

plt.xlabel("Bias to directly observed data, m")
plt.ylabel("Interpolation type number")
plt.legend(title="Region number", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(
    snakemake_fallback("output", "interp_bias_fig", str(project_path("figures", "figA1__bias_by_interp_type.pdf")))
)
plt.show()

In [ ]:
correlation_matrix = pd.DataFrame(
    data=[
        [diff.loc[:, :, i].corr(diff.loc[:, :, j]) for i in range(1, 6)]
        for j in range(1, 6)
    ],
    index=range(1, 6),
    columns=range(1, 6),
)
correlation_matrix = correlation_matrix.rename_axis(
    "elev_diff_interp_flag"
).rename_axis("elev_diff_interp_flag", axis=1)
correlation_matrix.to_csv(
    snakemake_fallback(
        "output", "corr_tbl", str(project_path("results", "uncertainty_by_interp__correlations.csv"))
    )
)
correlation_matrix

In [ ]:
uncertainty = diff.groupby(level="flag").apply(
    lambda x: x.quantile([0.25, 0.75]).diff()
)
uncertainty = uncertainty.droplevel(level=1).dropna().drop(0)
uncertainty = uncertainty.rename_axis("elev_diff_interp_flag").rename("uncertainty")
uncertainty = uncertainty / _norm_isf_25  # 2-sigma uncertainty
uncertainty.to_csv(
    snakemake_fallback(
        "output", "unc_tbl", str(project_path("results", "uncertainty_by_interp__uncertainty.csv"))
    )
)
uncertainty